In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from ipywidgets import interact
%matplotlib inline

def G(num, den, omega):
    """Evaluate G(jω) for a transfer function with polynomial coefficients.
    num, den: coefficient lists (highest power first), e.g. [1,2] = s+2
    omega: array of frequencies [rad/s]
    """
    s = 1j * np.asarray(omega, dtype=complex)
    return np.polyval(num, s) / np.polyval(den, s)

print("✓ G(num, den, omega) loaded — evaluates any rational transfer function on the imaginary axis.")


✓ G(num, den, omega) loaded — evaluates any rational transfer function on the imaginary axis.


## From Transfer Function to Complex Plane

A transfer function $G(s)$ is defined in the Laplace domain. To study frequency response, substitute $s = j\omega$:

$$G(j\omega) = \underbrace{|G(j\omega)|}_{\text{gain}} \cdot e^{\,j\angle G(j\omega)}$$

Every frequency $\omega$ maps to **one complex number** — a point in the plane with coordinates $\bigl(\operatorname{Re}[G(j\omega)],\; \operatorname{Im}[G(j\omega)]\bigr)$.  
Sweeping $\omega$ from $0$ to $+\infty$ traces the **Nyquist curve**. The mirror image (dashed) covers $\omega < 0$.

| Quantity | Meaning |
|---|---|
| $\omega = 0$ | Starting point — equals the DC gain $G(0)$ |
| $\omega \to \infty$ | End point — usually the origin for proper systems |
| $|G(j\omega)|$ | How much the system amplifies at that frequency |
| $\angle G(j\omega)$ | How much the system delays the phase at that frequency |


In [2]:
SYSTEMS = {
    'K/(s+1)      [1st order]':  ([1], [1, 1]),
    'K/(s+1)²     [repeated pole]': ([1], [1, 2, 1]),
    '1/(s²+0.5s+1)[2nd order underdamped]': ([1], [1, 0.5, 1]),
    '1/(s(s+1))   [integrator]': ([1], [1, 1, 0]),
}

def plot_phasor(system, log_omega):
    num, den = SYSTEMS[system]
    omega_val = 10**log_omega
    omega = np.logspace(-2, 3, 3000)

    # Avoid division by zero for integrator at omega=0
    omega_safe = omega[omega > 1e-6]
    Gjw = G(num, den, omega_safe)
    Gpt = G(num, den, [omega_val])[0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: phasor at current omega
    mag, ang = abs(Gpt), np.angle(Gpt, deg=True)
    ax1.annotate('', xy=(Gpt.real, Gpt.imag), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='crimson', lw=2.5))
    ax1.plot(Gpt.real, Gpt.imag, 'o', color='crimson', ms=9, zorder=5)
    lim = max(1.5, mag * 1.4)
    if mag > 0.05:
        arc_r = min(0.35 * mag, 0.35 * lim)
        theta_arc = np.linspace(0, np.angle(Gpt), 80)
        ax1.plot(arc_r * np.cos(theta_arc), arc_r * np.sin(theta_arc), 'b-', lw=1.8)
        mid = np.angle(Gpt) / 2
        ax1.text(arc_r * 1.2 * np.cos(mid), arc_r * 1.2 * np.sin(mid),
                 f'{ang:.0f}°', color='blue', fontsize=9, ha='center')
    ax1.axhline(0, color='k', lw=0.7); ax1.axvline(0, color='k', lw=0.7)
    ax1.set_xlim(-lim, lim); ax1.set_ylim(-lim, lim)
    ax1.set_aspect('equal'); ax1.grid(True, alpha=0.35)
    ax1.set_xlabel('Re[G(jω)]'); ax1.set_ylabel('Im[G(jω)]')
    ax1.set_title(f'Phasor at ω = {omega_val:.3f} rad/s\n|G| = {mag:.3f},  ∠G = {ang:.1f}°')

    # Right: full Nyquist + current point
    ax2.plot(Gjw.real, Gjw.imag, color='steelblue', lw=2.5, label='ω: 0 → +∞')
    ax2.plot(Gjw.real, -Gjw.imag, '--', color='steelblue', lw=1.5, alpha=0.4, label='ω: 0 → −∞')
    ax2.plot(Gpt.real, Gpt.imag, 'o', color='crimson', ms=11, zorder=5,
             label=f'ω = {omega_val:.3f}')
    ax2.plot(-1, 0, 'k+', ms=14, mew=2.5, label='(−1, 0)')
    ax2.axhline(0, color='k', lw=0.7); ax2.axvline(0, color='k', lw=0.7)
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.35)
    ax2.set_xlabel('Re[G(jω)]'); ax2.set_ylabel('Im[G(jω)]')
    ax2.set_title('Nyquist Curve — red dot tracks the slider')
    ax2.legend(fontsize=8)
    pad = 0.4
    finite = Gjw[np.isfinite(Gjw.real) & np.isfinite(Gjw.imag)]
    if len(finite):
        all_re = np.append(finite.real, [-1, Gpt.real])
        all_im = np.append(finite.imag, [0, Gpt.imag])
        ax2.set_xlim(np.nanmin(all_re) - pad, np.nanmax(all_re) + pad)
        ax2.set_ylim(np.nanmin(all_im) - pad, np.nanmax(all_im) + pad)

    plt.tight_layout(); plt.show()

interact(plot_phasor,
         system=widgets.Dropdown(options=list(SYSTEMS.keys()), description='G(s):',
                                 style={'description_width': 'initial'}),
         log_omega=widgets.FloatSlider(min=-2, max=2.5, step=0.05, value=0,
                                       description='log₁₀(ω)', continuous_update=True,
                                       style={'description_width': 'initial'}))


interactive(children=(Dropdown(description='G(s):', options=('K/(s+1)      [1st order]', 'K/(s+1)²     [repeat…

<function __main__.plot_phasor(system, log_omega)>

## First-Order Systems — G(s) = K / (τs + 1)

Substituting $s = j\omega$ and rationalising:

$$G(j\omega) = \frac{K}{1 + \omega^2\tau^2} - j\frac{K\omega\tau}{1 + \omega^2\tau^2}$$

The curve traces a **perfect semicircle** in the lower half-plane:

| $\omega$ | Point on plot |
|---|---|
| $0$ | $(K,\; 0)$ — DC gain |
| $1/\tau$ | $(K/2,\; -K/2)$ — half-power corner |
| $\infty$ | $(0,\; 0)$ |

The semicircle has **center** $(K/2, 0)$ and **radius** $K/2$.  
$K$ scales the entire plot; $\tau$ only changes speed along the curve, not its shape.  
A first-order system **can never encircle** $(−1, 0)$, so it is always closed-loop stable for any $K > 0$.


In [ ]:
def plot_first_order(K, tau):
    omega = np.logspace(-3, 3, 4000)
    Gjw = G([K], [tau, 1], omega)

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Nyquist
    ax.plot(Gjw.real, Gjw.imag, 'steelblue', lw=2.5, label='ω: 0 → +∞')
    ax.plot(Gjw.real, -Gjw.imag, '--', color='steelblue', lw=2, alpha=0.4, label='ω: 0 → −∞')
    ax.plot(K, 0, 'go', ms=9, label=f'DC gain = {K:.1f}')
    ax.plot(0, 0, 'rs', ms=9, label='ω → ∞')
    ax.plot(-1, 0, 'k+', ms=14, mew=2.5, label='(−1, 0)')

    # Theoretical semicircle
    th = np.linspace(0, -np.pi, 200)
    ax.plot(K/2 + K/2 * np.cos(th), K/2 * np.sin(th), 'r--', lw=1.2, alpha=0.6, label='Semicircle')
    ax.annotate('', xy=(K, 0), xytext=(K/2, 0),
                arrowprops=dict(arrowstyle='<->', color='orange', lw=2))
    ax.text(3*K/4, 0.07, f'r = K/2 = {K/2:.2f}', color='darkorange', ha='center', fontsize=9)

    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    lim = max(K * 1.25, 1.6)
    ax.set_xlim(-0.5, lim); ax.set_ylim(-lim * 0.7, lim * 0.7)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.35)
    ax.set_xlabel('Re[G(jω)]'); ax.set_ylabel('Im[G(jω)]')
    ax.set_title(f'Nyquist: G(s) = K/(τs+1),  K={K:.1f},  τ={tau:.1f}')
    ax.legend(fontsize=8)

    # Magnitude & phase vs frequency
    mag = np.abs(Gjw)
    phase = np.unwrap(np.angle(Gjw)) * 180 / np.pi
    ax2b = ax2.twinx()
    l1, = ax2.semilogx(omega, mag, 'steelblue', lw=2.2, label='|G(jω)|')
    l2, = ax2b.semilogx(omega, phase, 'orange', lw=2.2, ls='--', label='∠G [°]')
    ax2.axhline(1/np.sqrt(2), color='gray', ls=':', lw=1.2)
    ax2.axvline(1/tau, color='purple', ls=':', lw=1.2)
    ax2.text(1/tau * 1.05, 0.1, f'ωc=1/τ={1/tau:.2f}', color='purple', fontsize=8)
    ax2.set_xlabel('ω [rad/s]'); ax2.set_ylabel('|G(jω)|', color='steelblue')
    ax2b.set_ylabel('∠G(jω) [°]', color='orange')
    ax2b.set_ylim(-95, 5)
    ax2.set_title('Magnitude & Phase')
    ax2.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=8)
    ax2.grid(True, alpha=0.35)

    plt.tight_layout(); plt.show()

interact(plot_first_order,
         K=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1.5, description='K'),
         tau=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1.0, description='τ'))


interactive(children=(FloatSlider(value=1.5, description='K', max=5.0, min=0.1), FloatSlider(value=1.0, descri…

<function __main__.plot_first_order(K, tau)>

## How Poles and Zeros Shape the Nyquist Curve

Each open-loop **pole** at $s = -p$ (real, $p > 0$) contributes $-90°$ of phase as $\omega \to \infty$.  
Each open-loop **zero** at $s = -z$ contributes $+90°$ of phase.

$$\text{Total phase shift at } \omega \to \infty:\quad -90° \times (n_p - n_z)$$

where $n_p$ and $n_z$ are the number of poles and zeros.

For a **second-order** system $G(s) = \dfrac{K\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$, the damping $\zeta$ controls how far the curve swings into the lower half-plane.

| $\zeta$ | Behaviour |
|---|---|
| $> 1$ | Overdamped — two distinct real poles, gentle curve |
| $= 1$ | Critically damped |
| $0 < \zeta < 1$ | Underdamped — curve dips deeply, can loop back |
| $\to 0$ | Nearly oscillatory — curve nearly touches imaginary axis at $-j\infty$ |

Adding a **zero** in the numerator injects counter-clockwise rotation, shrinking the overall phase lag and tightening the curve.


In [4]:
def plot_second_order(K, zeta, wn, z_val, add_zero):
    omega = np.logspace(-2, 2.5, 4000)
    den = [1, 2*zeta*wn, wn**2]
    num = [K * wn**2]
    if add_zero:
        # Zero at s = -z_val: multiply numerator by (s + z_val) -> (1/z_val·s + 1), normalised
        num = np.polymul([K * wn**2 / z_val, K * wn**2], [1])
        num = [K * wn**2 / z_val, K * wn**2]

    Gjw = G(num, den, omega)
    poles = np.roots(den)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # s-plane
    ax1.axhline(0, color='k', lw=0.6); ax1.axvline(0, color='k', lw=0.6)
    ax1.plot(poles.real, poles.imag, 'rx', ms=13, mew=2.5, label='Poles')
    if add_zero:
        ax1.plot(-z_val, 0, 'bo', ms=11, mfc='none', mew=2.5, label=f'Zero at s=−{z_val:.1f}')
    span = max(wn * 2.2, 2)
    ax1.set_xlim(-span, span); ax1.set_ylim(-span, span)
    ax1.set_xlabel('Re(s)'); ax1.set_ylabel('Im(s)')
    ax1.set_title('s-Plane'); ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.35); ax1.set_aspect('equal')

    # Nyquist
    finite = np.isfinite(Gjw.real) & np.isfinite(Gjw.imag)
    Gf = Gjw[finite]
    ax2.plot(Gf.real, Gf.imag, 'steelblue', lw=2.5, label='ω: 0→+∞')
    ax2.plot(Gf.real, -Gf.imag, '--', color='steelblue', lw=1.5, alpha=0.4, label='ω: 0→−∞')
    ax2.plot(Gf.real[0], Gf.imag[0], 'go', ms=9, label='ω ≈ 0')
    ax2.plot(-1, 0, 'k+', ms=14, mew=2.5, label='(−1, 0)')
    ax2.axhline(0, color='k', lw=0.6); ax2.axvline(0, color='k', lw=0.6)
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.35)
    ax2.set_xlabel('Re[G(jω)]'); ax2.set_ylabel('Im[G(jω)]')
    title = f'Nyquist  K={K:.1f}, ζ={zeta:.2f}, ωₙ={wn:.1f}'
    if add_zero: title += f'\nZero at s=−{z_val:.1f}'
    ax2.set_title(title); ax2.legend(fontsize=8)
    vals = np.abs(Gf)
    r = min(np.percentile(vals, 95) * 2.0, 15) if len(vals) else 3
    r = max(r, 1.5)
    ax2.set_xlim(-r * 1.1, r); ax2.set_ylim(-r, r)

    plt.tight_layout(); plt.show()

interact(plot_second_order,
         K=widgets.FloatSlider(min=0.1, max=8, step=0.1, value=1.0, description='K'),
         zeta=widgets.FloatSlider(min=0.05, max=2.0, step=0.05, value=0.5, description='ζ (damping)'),
         wn=widgets.FloatSlider(min=0.3, max=5, step=0.1, value=1.0, description='ωₙ'),
         z_val=widgets.FloatSlider(min=0.1, max=6, step=0.1, value=2.0, description='Zero at −z'),
         add_zero=widgets.Checkbox(value=False, description='Add numerator zero'))


interactive(children=(FloatSlider(value=1.0, description='K', max=8.0, min=0.1), FloatSlider(value=0.5, descri…

<function __main__.plot_second_order(K, zeta, wn, z_val, add_zero)>

## The Nyquist Stability Criterion

The closed-loop characteristic equation is $1 + L(s) = 0$, where $L(s) = G(s)H(s)$ is the **loop transfer function**.  
The Nyquist criterion relates **open-loop** properties to **closed-loop** stability without solving for closed-loop poles:

$$\boxed{Z = N + P}$$

| Symbol | Meaning |
|---|---|
| $Z$ | Closed-loop poles in the **right half-plane** (want $Z = 0$) |
| $N$ | Net **clockwise** encirclements of $(-1+j0)$ by the Nyquist curve |
| $P$ | Open-loop poles in the **right half-plane** |

**How to count $N$:** Draw any ray from $(-1, 0)$. Each time the curve crosses the ray going clockwise: $+1$. Counterclockwise: $-1$. Sum = $N$.

| $N$ | $P$ | $Z = N+P$ | Verdict |
|---|---|---|---|
| 0 | 0 | 0 | **Stable** |
| 1 | 0 | 1 | Unstable |
| −1 | 1 | 0 | **Stable** (uncommon) |
| 0 | 1 | 1 | Unstable |

For a stable open-loop system ($P = 0$), stability requires that the Nyquist curve **does not encircle** $(-1, 0)$.  
Increasing gain $K$ scales the curve outward — eventually it sweeps past the critical point.


In [ ]:
def winding_number(re, im, cx=-1.0, cy=0.0):
    """Count net clockwise encirclements of (cx, cy) by a closed curve."""
    angles = np.arctan2(im - cy, re - cx)
    dang = np.diff(np.unwrap(angles))
    return int(np.round(-np.sum(dang) / (2 * np.pi)))  # negative = clockwise convention

def plot_criterion(K, p1, p2, p3):
    omega = np.logspace(-4, 4, 12000)
    poles_val = [p1, p2, p3]
    den = np.polymul([1, -p1], np.polymul([1, -p2], [1, -p3]))
    num = [K]

    P_rhp = sum(1 for p in poles_val if p > 0)
    Gjw = G(num, den, omega)

    finite = np.isfinite(Gjw.real) & np.isfinite(Gjw.imag)
    Gf = Gjw[finite]

    # Full closed Nyquist contour
    re_full = np.concatenate([Gf.real, Gf.real[::-1]])
    im_full = np.concatenate([Gf.imag, -Gf.imag[::-1]])
    N = winding_number(re_full, im_full)
    Z = N + P_rhp

    stable = Z == 0
    verdict = f"{'✅ STABLE' if stable else '❌ UNSTABLE'}   (N={N}, P={P_rhp}, Z={Z})"
    color_v = 'green' if stable else 'crimson'

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(Gf.real, Gf.imag, 'steelblue', lw=2.5, label='ω > 0')
    ax.plot(Gf.real, -Gf.imag, '--', color='steelblue', lw=1.8, alpha=0.4, label='ω < 0')

    # Arrows to show traversal direction
    for frac in [0.2, 0.5, 0.75]:
        idx = int(frac * len(Gf))
        if idx + 1 < len(Gf):
            ax.annotate('', xy=(Gf.real[idx+1], Gf.imag[idx+1]),
                        xytext=(Gf.real[idx], Gf.imag[idx]),
                        arrowprops=dict(arrowstyle='->', color='navy', lw=1.8))

    # Encirclement ray (downward from -1)
    ymin = np.nanmin(Gf.imag) - 0.5
    ax.plot([-1, -1], [0, ymin], 'purple', ls=':', lw=2, label='Encirclement ray')

    # Critical point
    ax.plot(-1, 0, 'k+', ms=18, mew=3, zorder=6)
    ax.annotate('(−1, 0)', (-1, 0), xytext=(8, 10), textcoords='offset points',
                fontsize=10, fontweight='bold')

    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    ax.set_xlabel('Re[L(jω)]'); ax.set_ylabel('Im[L(jω)]')
    ax.set_title(
        f'G(s) = K / [(s−{p1:.1f})(s−{p2:.1f})(s−{p3:.1f})],  K={K:.1f}\n'
        + verdict, fontsize=11, color=color_v)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.35); ax.set_aspect('equal')

    vals = np.abs(Gf)
    r = min(np.percentile(vals, 92) * 2.5, 12) if len(vals) else 4
    r = max(r, 2.0)
    ax.set_xlim(-r * 1.2, r); ax.set_ylim(-r, r)
    plt.tight_layout(); plt.show()

interact(plot_criterion,
         K=widgets.FloatSlider(min=0.1, max=120, step=0.5, value=5.0, description='K',
                               style={'description_width':'initial'}),
         p1=widgets.FloatSlider(min=-5, max=1.5, step=0.1, value=-1.0, description='pole₁ location',
                                style={'description_width':'initial'}),
         p2=widgets.FloatSlider(min=-5, max=1.5, step=0.1, value=-2.0, description='pole₂ location',
                                style={'description_width':'initial'}),
         p3=widgets.FloatSlider(min=-5, max=1.5, step=0.1, value=-3.0, description='pole₃ location',
                                style={'description_width':'initial'}))


interactive(children=(FloatSlider(value=5.0, description='K', max=120.0, min=0.1, step=0.5, style=SliderStyle(…

<function __main__.plot_criterion(K, p1, p2, p3)>

## Gain Margin and Phase Margin

Two scalar measures that quantify **how far from instability** a system is:

**Gain Margin (GM):** The factor by which $K$ can be multiplied before the Nyquist curve passes through $(-1, 0)$.  
It is measured at the **phase crossover frequency** $\omega_{pc}$ where $\angle G(j\omega_{pc}) = -180°$:

$$GM = \frac{1}{|G(j\omega_{pc})|}\qquad [\text{or}\quad GM_{\text{dB}} = -20\log_{10}|G(j\omega_{pc})|]$$

**Phase Margin (PM):** The additional phase lag (degrees) needed to make the system marginally stable.  
It is measured at the **gain crossover frequency** $\omega_{gc}$ where $|G(j\omega_{gc})| = 1$ (unit circle):

$$PM = 180° + \angle G(j\omega_{gc})$$

On the Nyquist plot:
- $GM$ appears as the **distance** from the curve's negative-real crossing to the critical point.
- $PM$ appears as the **angle** between the curve's unit-circle crossing and the $-180°$ direction.

Typical targets: $GM > 6\,\text{dB}$ and $PM > 30°$. Both margins should be positive for stability.


In [ ]:
def plot_margins(K, a, b, c):
    omega = np.logspace(-3, 3, 15000)
    den = np.polymul([1, a], np.polymul([1, b], [1, c]))
    num = [K]
    Gjw = G(num, den, omega)

    mag = np.abs(Gjw)
    phase_deg = np.unwrap(np.angle(Gjw)) * 180 / np.pi

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.plot(Gjw.real, Gjw.imag, 'steelblue', lw=2.5, label='Nyquist ω>0')
    ax.plot(Gjw.real, -Gjw.imag, '--', color='steelblue', lw=1.5, alpha=0.35, label='ω<0')

    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), color='gray', lw=1, ls=':', alpha=0.6, label='Unit circle')
    ax.plot(-1, 0, 'k+', ms=16, mew=3, zorder=6)
    ax.annotate('(−1, 0)', (-1, 0), xytext=(8, 10), textcoords='offset points',
                fontsize=10, fontweight='bold')

    gm_str = pm_str = 'N/A'

    # Gain crossover
    idx_gc = np.where(np.diff(np.sign(mag - 1.0)))[0]
    if len(idx_gc):
        i = idx_gc[0]
        frac = (1.0 - mag[i]) / (mag[i+1] - mag[i] + 1e-30)
        wgc = omega[i] + frac * (omega[i+1] - omega[i])
        Ggc = G(num, den, [wgc])[0]
        pm_val = 180 + np.angle(Ggc, deg=True)
        pm_str = f'{pm_val:.1f}°'
        ax.plot(Ggc.real, Ggc.imag, 'go', ms=13, zorder=5,
                label=f'Gain crossover  ωgc={wgc:.3f} rad/s')
        ang_gc = np.angle(Ggc)
        arc = np.linspace(-np.pi, ang_gc, 120)
        ax.plot(0.35*np.cos(arc), 0.35*np.sin(arc), 'g-', lw=2.5)
        mid_ang = (-np.pi + ang_gc) / 2
        ax.text(0.58*np.cos(mid_ang), 0.58*np.sin(mid_ang),
                f'PM = {pm_str}', color='green', fontsize=10, fontweight='bold', ha='center')

    # Phase crossover
    idx_pc = np.where(np.diff(np.sign(phase_deg + 180)))[0]
    if len(idx_pc):
        i = idx_pc[0]
        frac = (-180 - phase_deg[i]) / (phase_deg[i+1] - phase_deg[i] + 1e-30)
        wpc = omega[i] + frac * (omega[i+1] - omega[i])
        Gpc = G(num, den, [wpc])[0]
        mag_pc = abs(Gpc)
        if mag_pc > 1e-9:
            gm_val = 1.0 / mag_pc
            gm_str = f'{gm_val:.2f}x  ({20*np.log10(gm_val):.1f} dB)'
            ax.plot(Gpc.real, 0.0, 'rs', ms=13, zorder=5,
                    label=f'Phase crossover  ωpc={wpc:.3f} rad/s')
            ax.annotate('', xy=(-1, 0.06), xytext=(Gpc.real, 0.06),
                        arrowprops=dict(arrowstyle='<->', color='crimson', lw=2.5))
            xm = (Gpc.real - 1) / 2
            ax.text(xm, 0.18, f'GM = {gm_str}', color='crimson', fontsize=10,
                    fontweight='bold', ha='center')

    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    vals = np.abs(Gjw[np.isfinite(Gjw)])
    r = min(np.percentile(vals, 93)*2.5, 8) if len(vals) else 3
    r = max(r, 2.0)
    ax.set_xlim(-r, r); ax.set_ylim(-r, r)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.35)
    ax.set_xlabel('Re[G(jω)]'); ax.set_ylabel('Im[G(jω)]')
    ax.set_title(
        f'G(s) = K / [(s+{a:.1f})(s+{b:.1f})(s+{c:.1f})],  K={K:.1f}\n'
        f'GM = {gm_str}    PM = {pm_str}', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout(); plt.show()

interact(plot_margins,
         K=widgets.FloatSlider(min=0.1, max=150, step=0.5, value=6.0, description='K'),
         a=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1.0, description='pole −a'),
         b=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=2.0, description='pole −b'),
         c=widgets.FloatSlider(min=0.1, max=5, step=0.1, value=3.0, description='pole −c'))


interactive(children=(FloatSlider(value=6.0, description='K', max=150.0, min=0.1, step=0.5), FloatSlider(value…

<function __main__.plot_margins(K, a, b, c)>

## Time Delay — When the Curve Spirals

A pure time delay $e^{-Ls}$ is common in real systems (transport delay, computation lag, network latency, digital sampling $\approx T_s/2$).

Substituting $s = j\omega$:

$$e^{-Lj\omega} = \cos(L\omega) - j\sin(L\omega) \qquad \Rightarrow \qquad |e^{-jL\omega}| = 1, \quad \angle e^{-jL\omega} = -L\omega$$

The delay **does not change the magnitude** — it only **rotates each point by $-L\omega$ radians**.  
Since the rotation grows proportionally with $\omega$, the Nyquist curve **spirals inward**, crossing the negative real axis repeatedly.

Each additional spiral crossing of the unit circle at an angle past $-180°$ reduces the phase margin.  
Even a well-designed system with large margins can become unstable with small additional delay — this is why delay awareness is essential in digital control.

$$PM_{delay} \approx PM_0 - L \cdot \omega_{gc} \cdot \frac{180°}{\pi}$$


In [ ]:
def plot_delay(K, tau, L):
    omega = np.logspace(-2, 2.5, 6000)
    num, den = [K], [tau, 1]
    Gjw0 = G(num, den, omega)
    Gjw  = Gjw0 * np.exp(-1j * L * omega)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    theta = np.linspace(0, 2*np.pi, 300)

    for ax, data, title in [
        (ax1, Gjw0, 'No delay  (L = 0)'),
        (ax2, Gjw,  f'With delay  L = {L:.2f} s'),
    ]:
        ax.plot(data.real, data.imag, 'steelblue', lw=2.5, label='ω>0')
        ax.plot(data.real, -data.imag, '--', color='steelblue', lw=1.5, alpha=0.4, label='ω<0')
        ax.plot(data.real[0], data.imag[0], 'go', ms=9, label='ω≈0')
        ax.plot(np.cos(theta), np.sin(theta), 'gray', lw=1, ls=':', alpha=0.65)
        ax.plot(-1, 0, 'k+', ms=16, mew=3, label='(−1, 0)')
        ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
        ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.35)
        ax.set_xlabel('Re[G(jω)]'); ax.set_ylabel('Im[G(jω)]')
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8)

    # Compute and annotate PM for both
    for ax, data in [(ax1, Gjw0), (ax2, Gjw)]:
        mag = np.abs(data)
        idx = np.where(np.diff(np.sign(mag - 1.0)))[0]
        if len(idx):
            i = idx[0]
            frac = (1.0 - mag[i]) / (mag[i+1] - mag[i] + 1e-30)
            wgc = omega[i] + frac*(omega[i+1]-omega[i])
            Ggc_val = data[i] + frac*(data[i+1]-data[i])
            pm_val = 180 + np.angle(Ggc_val, deg=True)
            ax.plot(Ggc_val.real, Ggc_val.imag, 'ro', ms=11, zorder=5)
            arc = np.linspace(-np.pi, np.angle(Ggc_val), 100)
            ax.plot(0.3*np.cos(arc), 0.3*np.sin(arc), 'r-', lw=2)
            ax.text(0, -1.9, f'PM = {pm_val:.1f}°  (ωgc={wgc:.2f})', color='red',
                    ha='center', fontsize=10, fontweight='bold')

    plt.suptitle(
        f'G(s) = K·e^(−Ls) / (τs+1)    K={K:.1f},  τ={tau:.1f},  L={L:.2f}',
        fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

interact(plot_delay,
         K=widgets.FloatSlider(min=0.1, max=3.0, step=0.05, value=0.9, description='K'),
         tau=widgets.FloatSlider(min=0.1, max=5.0, step=0.1,  value=1.0, description='τ'),
         L=widgets.FloatSlider(min=0.0, max=10.0, step=0.1,  value=0.0, description='Delay L [s]'))


interactive(children=(FloatSlider(value=0.9, description='K', max=3.0, min=0.1, step=0.05), FloatSlider(value=…

<function __main__.plot_delay(K, tau, L)>